## Stream Customers Data From Cloud Files to Delta Lake using Auto Loader

1. Read files from cloud storage using Auto L
2. Transform the dataframe to add the following columns
    - file path: Cloud file path
    - ingestion date: Current Timestamp
3. Write the transformed data stream to Delta Lake Table

### Schema evolution mode

[Schema inference and evolution](https://learn.microsoft.com/en-us/azure/databricks/ingestion/cloud-object-storage/auto-loader/schema)

Autoloader can infer the schema from data that's being ingested into our data lake house.
Auto Loader supports the following modes for schema evolution, which you set in the cloudFiles.schemaEvolutionMode option:

| Mode | Behavior on reading new column |
|------|--------------------------------|
| `addNewColumns` (default) | Stream fails with an `UnknownFieldException` after Auto Loader adds the new columns to the schema. Restarting the stream resumes processing with the updated schema. Existing columns do not evolve data types. Azure Databricks recommends configuring Auto Loader streams with Lakeflow Jobs so they restart automatically. |
| `addNewColumnsWithTypeWidening` | Same behavior as `addNewColumns`, but Auto Loader also widens supported data types (such as `int` to `long`). Unsupported type changes (for example, `int` to `string`) are added to the [rescued data column](#). |
| `rescue` | Auto Loader never evolves the schema and the stream does not fail due to schema changes. Auto Loader records all new columns in the [rescued data column](#). |
| `failOnNewColumns` | Stream fails and does not restart unless you update the provided schema or remove the offending data file. The schema is not updated automatically. |
| `none` | Does not evolve the schema, new columns are ignored, and data is not rescued unless the `rescuedDataColumn` option is set. Stream does not fail due to schema changes. |


### 1. Read files using Autoloader

In [0]:
%sql
drop table if exists gizmobox.bronze.customers_autoloader;

In [0]:
customers_df = (
        spark.readStream
        .format("cloudFiles")
        .option("cloudFiles.format", "json")
        .option("cloudFiles.schemaLocation", "/Volumes/gizmobox/landing/operational_data/customers_autoloader/_schema")
        .option("cloudFiles.inferColumnTypes","true")
        .option("cloudFiles.schemaHints", "date_of_birth DATE, member_since DATE, created_timestamp TIMESTAMP")
        .option("cloudFiles.schemaEvolutionMode", "rescue")
        .load("/Volumes/gizmobox/landing/operational_data/customers_autoloader/")
)

### 2. Transform the dataframe to add the following columns

- file path: Cloud file path
- ingestion date: Current Timestamp

In [0]:
from pyspark.sql.functions import current_timestamp, col
customers_transformed_df =(
    customers_df.withColumn("file_path", col("_metadata.file_path"))
                .withColumn("ingestion_date", current_timestamp())
)

### 3. Write the transformed data stream to Delta Table

we need to put `.option("mergeSchema", "true")`

In [0]:
streaming_query =(
    customers_transformed_df.writeStream 
    .format("delta")
    .option("checkpointLocation", "/Volumes/gizmobox/landing/operational_data/customers_autoloader/_checkpoint_autoloader")
    .option("mergeSchema", "true")
    .toTable("gizmobox.bronze.customers_autoloader")
)

In [0]:
# para parar el streaming
streaming_query.stop()

In [0]:
%sql
select * from gizmobox.bronze.customers_autoloader;